# 08 — Wildfire Modeling

**Phase 4, Step 2.** Trains and compares classifiers and regressors on the
feature table from notebook 07.

**Input:**  `data/processed/wildfire_features.csv`
**Outputs:**
- `models/wildfire/classifier/<algo>.joblib`
- `models/wildfire/regressor/<algo>.joblib`
- `models/wildfire/summary.csv`, `best.json`

---

## Two complementary models

| Task | Target | Metric | Use |
|---|---|---|---|
| **Classification** | `fire_occurred` (binary) | ROC-AUC, PR-AUC, F1 | "Will there be ANY fire tomorrow near this city?" |
| **Regression** | `fire_count` (non-neg integer) | MAE, RMSE, Spearman ρ | "How many fire detections should we expect?" |

Both models share the same predictor set (weather + derived fire features +
NDVI + human activity + lightning + city dummies).

## Key design choices

- **Time-ordered split.** Test = 2024 (the last full year of FIRMS data).
- **Class imbalance** (~14% positive) handled by:
  - `class_weight="balanced"` in Logistic and HGBC
  - PR-AUC + F1 at optimal threshold reported alongside ROC-AUC
- **Probability calibration** via isotonic `CalibratedClassifierCV` on HGBC
  so `.predict_proba()` returns trustworthy risk values.
- **Count target** trained with `log1p` transform (Poisson-ish distribution)
  then `expm1`-ed back.
- **Baselines matter.** We report a random/strategic baseline so the improvement
  of trained models is demonstrable, not assumed.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

from src.wildfire import train as wft
from src.utils.config import MODELS_DIR, PROCESSED_DIR
from src.utils.logging_utils import get_logger
logger = get_logger("nb.08_wildfire_modeling")

## 2. Inspect the training data

In [ ]:
df = pd.read_csv(PROCESSED_DIR / "wildfire_features.csv", parse_dates=["date"])
print(f"Rows: {len(df):,}, columns: {df.shape[1]}")

labelled = df.dropna(subset=["fire_occurred"])
print(f"Labelled rows: {len(labelled):,} ({labelled['fire_occurred'].mean():.1%} positive)")
print(f"Date range of labels: {labelled['date'].min().date()} -> {labelled['date'].max().date()}")

## 3. Prepare the time-ordered split

Train = before 2024-01-01, test = 2024 (the last full FIRMS year).

In [ ]:
split_clf = wft.prepare_split(df, "fire_occurred", test_start="2024-01-01")
split_reg = wft.prepare_split(df, "fire_count",    test_start="2024-01-01")

print(f"Classifier split:  train {split_clf.X_train.shape}, test {split_clf.X_test.shape}")
print(f"                   train pos rate = {split_clf.y_train.mean():.3f},  test pos rate = {split_clf.y_test.mean():.3f}")
print(f"Regressor  split:  train {split_reg.X_train.shape}, test {split_reg.X_test.shape}")
print(f"                   train mean count = {split_reg.y_train.mean():.3f}, test mean count = {split_reg.y_test.mean():.3f}")

## 4. Train classifiers and compare

In [ ]:
clf_results = {}
for algo in ["baseline", "logistic", "hgbc"]:
    model, scores, metrics = wft.train_fire_classifier(split_clf, algo=algo)
    clf_results[algo] = {"model": model, "scores": scores, "metrics": metrics}

pd.DataFrame({a: r["metrics"] for a, r in clf_results.items()}).round(3)

### 4a. ROC and Precision-Recall curves

Two different views of classifier quality. PR curve is more informative for
imbalanced targets like ours (14% positive).

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for algo, r in clf_results.items():
    if algo == "baseline": continue
    scores = r["scores"]
    y = split_clf.y_test.values

    fpr, tpr, _ = roc_curve(y, scores)
    axes[0].plot(fpr, tpr, lw=1.5,
                 label=f"{algo} (AUC={r['metrics']['ROC_AUC']:.3f})")

    prec, rec, _ = precision_recall_curve(y, scores)
    axes[1].plot(rec, prec, lw=1.5,
                 label=f"{algo} (PR-AUC={r['metrics']['PR_AUC']:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5, label="chance")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC curve"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].axhline(split_clf.y_test.mean(), ls="--", color="k", lw=0.8, alpha=0.5,
                label=f"baseline prevalence = {split_clf.y_test.mean():.2f}")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("Precision-Recall curve"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

### 4b. Threshold selection and confusion matrix

At the F1-optimal threshold, how does the best classifier confuse fire / no-fire?

In [ ]:
from sklearn.metrics import confusion_matrix

best_algo = max(clf_results, key=lambda a: clf_results[a]["metrics"].get("PR_AUC", -1)
                if a != "baseline" else -1)
best = clf_results[best_algo]
y_true = split_clf.y_test.values.astype(int)
y_pred = (best["scores"] >= best["metrics"]["best_threshold"]).astype(int)
cm = confusion_matrix(y_true, y_pred)

print(f"Best classifier: {best_algo}")
print(f"Threshold      : {best['metrics']['best_threshold']:.3f}")
print(f"Confusion matrix (rows=actual, cols=predicted):")
print(f"                 no-fire   fire")
print(f"  actual no-fire  {cm[0,0]:6d}  {cm[0,1]:5d}")
print(f"  actual fire     {cm[1,0]:6d}  {cm[1,1]:5d}")
print(f"\nPrecision = {cm[1,1] / max(cm[1,1]+cm[0,1], 1):.3f}")
print(f"Recall    = {cm[1,1] / max(cm[1,1]+cm[1,0], 1):.3f}")

## 5. Train count regressors and compare

In [ ]:
reg_results = {}
for algo in ["baseline", "ridge", "hgbr_log"]:
    model, preds, metrics = wft.train_fire_count_regressor(split_reg, algo=algo)
    reg_results[algo] = {"model": model, "preds": preds, "metrics": metrics}

pd.DataFrame({a: r["metrics"] for a, r in reg_results.items()}).round(3)

### 5a. Predicted vs actual count (best regressor)

Perfect prediction would lie on `y = x`. For rare events, we expect the model
to under-predict large counts (regression to the mean) — that's OK as long
as the ranking (Spearman ρ) is preserved.

In [ ]:
best_reg_algo = min(reg_results, key=lambda a: reg_results[a]["metrics"]["MAE"]
                    if a != "baseline" else np.inf)
best_reg = reg_results[best_reg_algo]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(split_reg.y_test, best_reg["preds"], s=8, alpha=0.35, c="steelblue")
mn = 0; mx = max(split_reg.y_test.max(), best_reg["preds"].max())
ax.plot([mn, mx], [mn, mx], "k-", lw=1, alpha=0.6, label="y = x")
ax.set_xlabel("actual fire_count"); ax.set_ylabel("predicted fire_count")
ax.set_title(f"{best_reg_algo}  —  Spearman ρ = {best_reg['metrics']['spearman']:.3f}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Feature importance for the classifier

For the logistic model, coefficient × standardised feature scale gives a
direct comparable weight. For HGBC we use permutation importance.

In [ ]:
from sklearn.inspection import permutation_importance

# Use HGBC for permutation importance if present, else logistic coefficients
if "hgbc" in clf_results:
    model = clf_results["hgbc"]["model"]
    # Subsample test set for speed
    rng = np.random.default_rng(0)
    idx = rng.choice(len(split_clf.X_test), size=min(500, len(split_clf.X_test)), replace=False)
    imp = permutation_importance(
        model, split_clf.X_test.iloc[idx], split_clf.y_test.iloc[idx],
        n_repeats=5, random_state=0, scoring="roc_auc", n_jobs=1,
    )
    fi = pd.DataFrame({
        "feature": split_clf.X_test.columns,
        "importance": imp.importances_mean,
        "std": imp.importances_std,
    }).sort_values("importance", ascending=False).head(15)
    print("HGBC permutation importance (ROC-AUC drop):")
    print(fi.to_string(index=False))
elif "logistic" in clf_results:
    bundle = clf_results["logistic"]["model"]
    coef = bundle["clf"].coef_[0]
    # |coef| is already roughly comparable because features were standardised
    fi = pd.DataFrame({"feature": split_clf.X_test.columns,
                       "coef": coef,
                       "abs_coef": np.abs(coef)})
    print("Logistic coefficients (top 15 by |coef|):")
    print(fi.sort_values("abs_coef", ascending=False).head(15).to_string(index=False))

## 7. Full orchestrator

Trains every configured algo (incl. XGBoost if installed) and persists a summary.

In [ ]:
out = wft.train_wildfire_models(
    classifier_algos=["baseline", "logistic", "hgbc", "xgboost"],  # xgboost auto-skips if absent
    regressor_algos =["baseline", "ridge", "hgbr_log"],
)
print("=== SUMMARY ===")
out["summary"].round(3)

### 7a. Best model per task (from `best.json`)

In [ ]:
best = json.loads((MODELS_DIR / "wildfire" / "best.json").read_text())
for task, d in best.items():
    print(f"\n{task}:  algo={d['algo']}  metric={d['metric']:.3f}")
    print(f"  model_path = {d['path']}")
    for k, v in (d.get('metrics') or {}).items():
        if isinstance(v, (int, float)) and v is not None:
            print(f"  {k:18s} {v:.3f}" if isinstance(v, float) else f"  {k:18s} {v}")

## 8. Caveats

- **Small test set** (~1,830 labeled rows in 2024). PR-AUC confidence intervals
  are wide. A second year of data will likely shift the winning algo.
- **Label noise.** FIRMS hotspots include some false positives
  (e.g. hot industrial surfaces, especially in Baku). We filtered by
  `type=0, confidence!='l'`, but residual noise is unavoidable.
- **No cross-city leakage test.** The pooled model uses city dummies — we did
  not evaluate held-out-city generalisation. If you plan to deploy to
  cities not in the training set, repeat with a leave-one-city-out split.
- **Baseline rate varies by city** (8% Zaqatala vs 27% Baku). Per-city
  recalibration would be a natural next step.

## 9. Persisted artefacts

In [ ]:
for p in sorted((MODELS_DIR / "wildfire").rglob("*")):
    if p.is_file():
        rel = p.relative_to(MODELS_DIR)
        print(f"  {str(rel):50s} {p.stat().st_size/1024:8.2f} KB")

---
**Step 2 of Phase 4 ✅ complete.** Next: `09_wildfire_prediction.ipynb` — apply
the classifier + regressor to the 30-day weather forecast to produce a forward
wildfire risk dataset.